In [1]:
# Notebook 2 — Pattern extraction and guided extended FP-Growth (Apriori prototype)
# This notebook:
# 1) Loads cleaned descriptions from Notebook 1
# 2) Extracts adjective–noun (and noun–noun) patterns (multilingual-aware)
# 3) Builds transactions for frequent itemset mining
# 4) Runs a guided FP Growth (interpretable, small-scale) to find frequent itemsets
# 5) Generates OSM-targeted association rules
# 6) Saves artifacts and a short Markdown report

import os
import re
import json
import math
import random
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# IO paths
DATA_PATH = "data/processed/descriptions_clean.csv"
RESULTS_DIR = "results"
REPORTS_DIR = "docs/reports"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Mining configuration (tune as needed)
MIN_SUP = 0.05          # minimum support (fraction of transactions)
MIN_CONF = 0.5          # minimum confidence for rules
MAX_ITEMSET_LEN = 3     # keep itemsets small for interpretability
TOP_K_ITEMS_FALLBACK = 200  # cap the item vocabulary if data is large, to keep it fast

TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
print("Config ready. Seed set. Paths OK.")

Config ready. Seed set. Paths OK.


C:\Users\stran\AppData\Local\Temp\ipykernel_4952\2723117059.py:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")


In [4]:
# Load the dataset produced in 02_data_audit_schema_alignment.ipynb
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"{DATA_PATH} not found. Please run Notebook 1 first.")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "rows:", len(df))

Loaded: data/processed/descriptions_clean.csv rows: 3


In [5]:
df.head()

,id,osm_id,osm_geom_type,osm_tag_key,osm_tag_value,name,name_ur,alt_name,description_raw,description_final,description_source,wikipedia_title,wikipedia_url,lat,lon,city,province,country,language,dedup_group_id
0,toy-1,n1,node,amenity,park,Kids Park F-7,بچوں کا پارک ایف-7,NaN,"A quiet park near the F-7 markaz, perfect for ...","A quiet park near the F-7 markaz, perfect for ...",toy,NaN,NaN,33.723,73.055,Islamabad,ICT,Pakistan,Roman Urdu,0dc2b872-da697349
1,toy-2,w2,way,amenity,place_of_worship,Masjid-e-Quba,مسجد قبا,Quba Mosque,پرانے بازار کے قریب ایک خوبصورت مسجد۔,پرانے بازار کے قریب ایک خوبصورت مسجد۔,toy,NaN,NaN,33.706,73.039,Islamabad,ICT,Pakistan,Urdu,05f94746-e0e4091e
2,toy-3,n3,node,shop,mall,Centaurus,سینٹورس,The Centaurus Mall,Famous mall with food court and cinema.,Famous mall with food court and cinema.,toy,The Centaurus,NaN,33.710,73.058,Islamabad,ICT,Pakistan,English,b6ece0a2-82d7bdcf


In [6]:
# Expect these columns from 02_data_audit_schema_alignment.ipynb
expected_cols = [
    "id","osm_tag_key","osm_tag_value","name","name_ur","alt_name",
    "description_final","language","city","province","country","lat","lon"
]
missing_cols = [c for c in expected_cols if c not in df.columns]
if missing_cols:
    print("Warning: missing expected columns:", missing_cols)

# Combine OSM tag as label (target)
df["osm_label"] = df["osm_tag_key"].fillna("") + "=" + df["osm_tag_value"].fillna("")
print("Unique osm_label count:", df["osm_label"].nunique())
df.head()

Unique osm_label count: 3


,id,osm_id,osm_geom_type,osm_tag_key,osm_tag_value,name,name_ur,alt_name,description_raw,description_final,...,wikipedia_title,wikipedia_url,lat,lon,city,province,country,language,dedup_group_id,osm_label
0,toy-1,n1,node,amenity,park,Kids Park F-7,بچوں کا پارک ایف-7,NaN,"A quiet park near the F-7 markaz, perfect for ...","A quiet park near the F-7 markaz, perfect for ...",...,NaN,NaN,33.723,73.055,Islamabad,ICT,Pakistan,Roman Urdu,0dc2b872-da697349,amenity=park
1,toy-2,w2,way,amenity,place_of_worship,Masjid-e-Quba,مسجد قبا,Quba Mosque,پرانے بازار کے قریب ایک خوبصورت مسجد۔,پرانے بازار کے قریب ایک خوبصورت مسجد۔,...,NaN,NaN,33.706,73.039,Islamabad,ICT,Pakistan,Urdu,05f94746-e0e4091e,amenity=place_of_worship
2,toy-3,n3,node,shop,mall,Centaurus,سینٹورس,The Centaurus Mall,Famous mall with food court and cinema.,Famous mall with food court and cinema.,...,The Centaurus,NaN,33.710,73.058,Islamabad,ICT,Pakistan,English,b6ece0a2-82d7bdcf,shop=mall


In [7]:
# We use NLTK for English POS tagging (simple and lightweight)
# For Urdu and Roman Urdu, we'll use heuristic patterns and curated lexicons.
import nltk

# Download resources if missing (safe to re-run)
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

try:
    nltk.data.find("taggers/averaged_perceptron_tagger")
except LookupError:
    nltk.download("averaged_perceptron_tagger")

from nltk import word_tokenize, pos_tag
print("NLTK ready.")

NLTK ready.


In [8]:
# OSM-relevant Pakistan-centric lexicon to guide pattern mining
# These provide domain bias so we keep interpretable, useful items.
OSM_VALUE_SYNONYMS = {
    "amenity=mosque": {"mosque", "masjid", "jamia", "imam-bargah", "imambargah"},
    "amenity=school": {"school", "madrasa", "college"},
    "amenity=hospital": {"hospital", "clinic"},
    "amenity=park": {"park", "family-park", "kid-park"},
    "shop=mall": {"mall", "markaz", "center", "centaurus"},
    "amenity=bazaar": {"bazaar", "bazar", "mandi", "market"},
    "highway=chowk": {"chowk", "roundabout"},
    "amenity=restaurant": {"restaurant", "hotel", "dhaba", "eatery"},
    "amenity=cafe": {"cafe", "coffee"},
    "amenity=bank": {"bank", "atm"},
    "amenity=university": {"university", "uni", "campus"},
}

# Adjective cues (English + Roman Urdu common transliterations)
ADJECTIVE_CUES = {
    "quiet","peaceful","beautiful","old","new","historic","famous","popular","big","small",
    "family","kids","cheap","expensive","crowded","busy","clean","green",
    "khubsurat","purana","naya","bari","choti","mashhoor"
}

# Noun cues (English + Roman Urdu common transliterations)
NOUN_CUES = {
    "park","mosque","masjid","bazaar","bazar","market","mandi","chowk","roundabout","mall","markaz",
    "school","college","university","hospital","clinic","restaurant","cafe","bank",
    "lake","trail","museum","cinema","zoo"
}

# Urdu script cues for nouns (minimal; mostly to capture obvious ones)
URDU_NOUN_CUES = {
    "مسجد","بازار","چوک","پارک","اسکول","کالج","ہسپتال","کلیہ","ریسٹورنٹ","بینک","چڑیاگھر","سینما"
}

print("Lexicons loaded: values", len(OSM_VALUE_SYNONYMS), "adj", len(ADJECTIVE_CUES), "noun", len(NOUN_CUES))

Lexicons loaded: values 11 adj 24 noun 24


In [9]:
# Cleaners, tokenizers, and language-aware pattern extraction helpers
# Basic text cleaner
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Tokenize English text with NLTK; fallback to split for others
def tokenize(text: str, lang: str) -> list:
    text = clean_text(text)
    if not text:
        return []
    if lang in ("English", "Roman Urdu", "Mixed"):  # Mixed may contain Latin words
        try:
            return word_tokenize(text)
        except Exception:
            return text.split()
    # Urdu or other: simple whitespace split (POS not available here)
    return text.split()

# POS tag only for English-ish tokens; otherwise return (token, 'X')
def pos_tag_safe(tokens: list, lang: str) -> list:
    if not tokens:
        return []
    if lang in ("English", "Roman Urdu", "Mixed"):
        try:
            return pos_tag(tokens)
        except Exception:
            return [(t, "X") for t in tokens]
    return [(t, "X") for t in tokens]

# Extract candidate adjective–noun and noun–noun patterns
# For English: use POS (JJ, JJR, JJS + NN/NNS/NNP etc.)
# For Roman Urdu/Urdu: use heuristic match against cue lists and proximity windows.
def extract_patterns(text: str, lang: str) -> set:
    patterns = set()
    toks = tokenize(text, lang)

    # Short-circuit if nothing
    if not toks:
        return patterns

    if lang == "English" or lang == "Mixed":
        tagged = pos_tag_safe(toks, lang)  # e.g., [('quiet', 'JJ'), ('park','NN')]
        # Adjective-Noun bigrams
        for i in range(len(tagged) - 1):
            w1, p1 = tagged[i]
            w2, p2 = tagged[i+1]
            if p1.startswith("JJ") and p2.startswith("NN"):
                patterns.add(f"{w1}_{w2}")  # adj_noun
        # Noun-Noun compounds
        for i in range(len(tagged) - 1):
            w1, p1 = tagged[i]
            w2, p2 = tagged[i+1]
            if p1.startswith("NN") and p2.startswith("NN"):
                patterns.add(f"{w1}_{w2}")  # noun_noun

        # Also include single-word noun/adj cues present
        for w, p in tagged:
            if w in ADJECTIVE_CUES:
                patterns.add(f"adj:{w}")
            if w in NOUN_CUES:
                patterns.add(f"noun:{w}")

    elif lang == "Roman Urdu":
        # Heuristic: scan sliding window of size 2-3 and match cues
        for i in range(len(toks) - 1):
            w1, w2 = toks[i], toks[i+1]
            if w1 in ADJECTIVE_CUES and w2 in NOUN_CUES:
                patterns.add(f"{w1}_{w2}")
        # Include single-word cue features
        for w in toks:
            if w in ADJECTIVE_CUES:
                patterns.add(f"adj:{w}")
            if w in NOUN_CUES:
                patterns.add(f"noun:{w}")

    elif lang == "Urdu":
        # Very simple heuristic: directly match Urdu noun cues and bigrams around them
        for i, w in enumerate(toks):
            if w in URDU_NOUN_CUES:
                patterns.add(f"noun_ur:{w}")
                # Include previous token as adjective if it's not punctuation and in a small Urdu adj set (not curated here)
                if i > 0 and re.match(r"^\w+$", toks[i-1]):
                    patterns.add(f"ur_prev_{toks[i-1]}_{w}")

    else:
        # Unknown: fallback — include common noun cues if present
        for w in toks:
            if w in NOUN_CUES:
                patterns.add(f"noun:{w}")

    return patterns

In [10]:
# Each "transaction" = set of items from one description, including:
# - Extracted patterns (adj_noun, noun_noun, and cue features)
# - Guided OSM items when synonyms appear (e.g., 'amenity=mosque' if "masjid" appears)
# - Optional city-level token to let city-specific patterns emerge (Islamabad)
transactions = []
labels = []  # the target label (OSM key=value)
meta_rows = []  # keep ids for traceability

for idx, row in df.iterrows():
    text = row.get("description_final", "") or ""
    lang = row.get("language", "Unknown")
    city = clean_text(row.get("city", "") or "")
    label = row.get("osm_label", "")
    rid = row.get("id", idx)

    items = set()
    # Extract patterns
    pats = extract_patterns(text, lang)
    items.update(pats)

    # Include city token (optional, helps locality-driven patterns)
    if city:
        items.add(f"city:{city}")

    # Guided OSM synonym mapping: if text contains any synonym, add the canonical OSM value item
    text_clean = clean_text(text)
    for canonical, syns in OSM_VALUE_SYNONYMS.items():
        if any(s in text_clean for s in syns):
            items.add(f"osm_hint:{canonical}")

    # Also include the surface OSM label as an item for rule learning (as consequent later)
    # But not for itemset frequency decisions (we won't let label guide itself unfairly)
    # We'll keep it separate in labels[], not in items.

    transactions.append(items)
    labels.append(label)
    meta_rows.append(rid)

print("Built transactions:", len(transactions))
# Quick peek: non-empty transaction rate
print("Non-empty transactions:", sum(1 for t in transactions if len(t) > 0))

Built transactions: 3
Non-empty transactions: 3


In [11]:
# Prune the item vocabulary (keep OSM-relevant + frequent items)
# To keep mining efficient and interpretable, prune the vocabulary:
# 1) Always keep items starting with 'osm_hint:'
# 2) Keep items that occur at least freq_min times
# 3) If still too many, keep top-k by frequency

N = len(transactions)
freq = Counter()
for t in transactions:
    freq.update(t)

# Minimum absolute frequency derived from MIN_SUP
freq_min = max(2, math.ceil(MIN_SUP * N))

# Items to keep
keep_items = set([it for it, c in freq.items() if c >= freq_min or it.startswith("osm_hint:")])

# If vocabulary is still too big, keep top-K frequent (but never drop osm_hint)
if len(keep_items) > TOP_K_ITEMS_FALLBACK:
    sorted_items = sorted([(it, freq[it]) for it in keep_items if not it.startswith("osm_hint:")],
                          key=lambda x: x[1], reverse=True)
    # Always include hints
    hints = [it for it in keep_items if it.startswith("osm_hint:")]
    top_items = [it for it, _ in sorted_items[:(TOP_K_ITEMS_FALLBACK - len(hints))]]
    keep_items = set(hints + top_items)

# Apply pruning to transactions
transactions_pruned = [set(it for it in t if it in keep_items) for t in transactions]

print("Original vocab size:", len(freq))
print("Kept vocab size:", len(keep_items))
print("Average items per transaction (before/after):",
      np.mean([len(t) for t in transactions]).round(2), "->",
      np.mean([len(t) for t in transactions_pruned]).round(2))

Original vocab size: 15
Kept vocab size: 3
Average items per transaction (before/after): 6.0 -> 2.0


In [12]:
# Simple Apriori implementation for small/medium datasets.
# Returns dict: itemset (tuple) -> support (fraction)

def apriori(transactions, min_sup=0.1, max_len=3):
    T = [set(t) for t in transactions if t]  # remove empties
    if not T:
        return {}

    N = len(T)
    sup_abs = math.ceil(min_sup * N)

    # Generate L1
    C1 = Counter()
    for t in T:
        for i in t:
            C1[i] += 1
    L1 = { (i,): c for i, c in C1.items() if c >= sup_abs }
    L_all = dict(L1)

    # Iteratively generate candidates
    Lk = L1
    k = 2
    while Lk and k <= max_len:
        prev_itemsets = list(Lk.keys())
        # Join step
        Ck_candidates = set()
        for i in range(len(prev_itemsets)):
            for j in range(i+1, len(prev_itemsets)):
                a = prev_itemsets[i]
                b = prev_itemsets[j]
                # join if first k-2 items equal (sorted tuples)
                if tuple(sorted(a))[:-1] == tuple(sorted(b))[:-1]:
                    candidate = tuple(sorted(set(a) | set(b)))
                    if len(candidate) == k:
                        Ck_candidates.add(candidate)

        # Prune step: all (k-1)-subsets must be frequent
        Lk_sets = set(prev_itemsets)
        pruned_candidates = []
        for cand in Ck_candidates:
            all_subsets_frequent = True
            # generate all (k-1) subsets
            for idx in range(len(cand)):
                subset = tuple(sorted(cand[:idx] + cand[idx+1:]))
                if subset not in Lk_sets:
                    all_subsets_frequent = False
                    break
            if all_subsets_frequent:
                pruned_candidates.append(cand)

        # Count support
        Ck_counts = Counter()
        for t in T:
            for cand in pruned_candidates:
                # subset check
                if all(x in t for x in cand):
                    Ck_counts[cand] += 1

        # Keep frequent
        Lk = { itemset: cnt for itemset, cnt in Ck_counts.items() if cnt >= sup_abs }
        L_all.update(Lk)
        k += 1

    # Convert counts to fractional support
    L_all_sup = { tuple(sorted(k)): v / N for k, v in L_all.items() }
    return L_all_sup

freq_itemsets = apriori(transactions_pruned, min_sup=MIN_SUP, max_len=MAX_ITEMSET_LEN)
print("Frequent itemsets found:", len(freq_itemsets))

# Convert to DataFrame for inspection
fis_rows = []
for items, sup in sorted(freq_itemsets.items(), key=lambda x: (-x[1], len(x[0]))):
    fis_rows.append({"itemset": items, "length": len(items), "support": round(sup, 4)})
fis_df = pd.DataFrame(fis_rows)
fis_df.head(10)

Frequent itemsets found: 7


,itemset,length,support
0,"(city:islamabad,)",1,1.0000
1,"(osm_hint:shop=mall,)",1,0.6667
2,"(city:islamabad, osm_hint:shop=mall)",2,0.6667
3,"(osm_hint:amenity=park,)",1,0.3333
4,"(osm_hint:amenity=park, osm_hint:shop=mall)",2,0.3333
5,"(city:islamabad, osm_hint:amenity=park)",2,0.3333
6,"(city:islamabad, osm_hint:amenity=park, osm_hi...",3,0.3333


In [13]:
# We aim for rules of the form: antecedent -> osm_label (key=value)
# We'll compute:
# - support: P(antecedent ∪ consequent)
# - confidence: P(consequent | antecedent)
# - lift: confidence / P(consequent)

# Prepare indices for fast membership checks
T = transactions_pruned
y = labels

# Support of labels
label_counts = Counter(y)
N = len(T)
label_support = {lab: c / N for lab, c in label_counts.items() if lab and lab != "="}

def rules_from_itemsets(freq_itemsets_dict, min_conf=0.5):
    rules = []
    # Map itemset -> support
    fis_sup = freq_itemsets_dict
    # Build index: for each itemset, which rows contain it (to compute conditional stats robustly)
    itemset_rows_cache = {}

    def rows_with_itemset(itemset):
        key = tuple(sorted(itemset))
        if key in itemset_rows_cache:
            return itemset_rows_cache[key]
        rows = []
        for i, t in enumerate(T):
            if all(x in t for x in itemset):
                rows.append(i)
        itemset_rows_cache[key] = rows
        return rows

    # Consider itemsets as potential antecedents if they don't already include an osm_label item
    # (We did not include labels into transactions; labels are separate y)
    for itemset, sup in fis_sup.items():
        if len(itemset) == 0:
            continue
        rows = rows_with_itemset(itemset)
        if not rows:
            continue

        # Distribution of labels given the antecedent
        cond_label_counts = Counter(y[i] for i in rows)
        total = sum(cond_label_counts.values())

        for lab, cnt in cond_label_counts.items():
            if not lab or lab == "=":
                continue
            conf = cnt / total if total else 0.0
            sup_joint = cnt / N
            base = label_support.get(lab, 1e-12)
            lift = conf / base if base > 0 else np.inf
            if conf >= min_conf:
                rules.append({
                    "antecedent": tuple(sorted(itemset)),
                    "consequent": lab,
                    "support": round(sup_joint, 4),
                    "confidence": round(conf, 4),
                    "lift": round(lift, 4),
                    "antecedent_len": len(itemset)
                })

    # Sort by lift then confidence
    rules = sorted(rules, key=lambda r: (-r["lift"], -r["confidence"], -r["support"]))
    return rules

rules = rules_from_itemsets(freq_itemsets, min_conf=MIN_CONF)
rules_df = pd.DataFrame(rules)
print("Rules generated:", len(rules_df))
rules_df.head(10)

Rules generated: 8


,antecedent,consequent,support,confidence,lift,antecedent_len
0,"(osm_hint:amenity=park,)",amenity=park,0.3333,1.0,3.0,1
1,"(osm_hint:amenity=park, osm_hint:shop=mall)",amenity=park,0.3333,1.0,3.0,2
2,"(city:islamabad, osm_hint:amenity=park)",amenity=park,0.3333,1.0,3.0,2
3,"(city:islamabad, osm_hint:amenity=park, osm_hi...",amenity=park,0.3333,1.0,3.0,3
4,"(osm_hint:shop=mall,)",amenity=park,0.3333,0.5,1.5,1
5,"(osm_hint:shop=mall,)",shop=mall,0.3333,0.5,1.5,1
6,"(city:islamabad, osm_hint:shop=mall)",amenity=park,0.3333,0.5,1.5,2
7,"(city:islamabad, osm_hint:shop=mall)",shop=mall,0.3333,0.5,1.5,2


In [14]:
# Save frequent itemsets and rules
fis_out = os.path.join(RESULTS_DIR, "frequent_itemsets.csv")
fis_json = os.path.join(RESULTS_DIR, "frequent_itemsets.json")
rules_out = os.path.join(RESULTS_DIR, "association_rules.csv")
rules_json = os.path.join(RESULTS_DIR, "association_rules.json")

fis_df.to_csv(fis_out, index=False)
fis_df.to_json(fis_json, orient="records", force_ascii=False, indent=2)

if len(rules_df) > 0:
    rules_df.to_csv(rules_out, index=False)
    rules_df.to_json(rules_json, orient="records", force_ascii=False, indent=2)
else:
    # Create empty placeholder files so downstream steps don't fail
    pd.DataFrame(columns=["antecedent","consequent","support","confidence","lift","antecedent_len"]).to_csv(rules_out, index=False)
    pd.DataFrame(columns=["antecedent","consequent","support","confidence","lift","antecedent_len"]).to_json(rules_json, orient="records", indent=2)

print("Saved itemsets to:", fis_out, "and", fis_json)
print("Saved rules to:", rules_out, "and", rules_json)

Saved itemsets to: results\frequent_itemsets.csv and results\frequent_itemsets.json
Saved rules to: results\association_rules.csv and results\association_rules.json


In [15]:
# Short normal report
report_md_path = os.path.join(REPORTS_DIR, f"patterns_mining_{datetime.utcnow().strftime('%Y%m%d')}.md")
with open(report_md_path, "w", encoding="utf-8") as f:
    f.write("### Notebook 2 — Pattern Extraction & Guided Frequent Itemsets\n\n")
    f.write(f"- Timestamp: {TIMESTAMP}\n")
    f.write(f"- Transactions: {len(transactions)} (after pruning: {len(transactions_pruned)})\n")
    f.write(f"- MIN_SUP: {MIN_SUP}, MIN_CONF: {MIN_CONF}, MAX_ITEMSET_LEN: {MAX_ITEMSET_LEN}\n")
    f.write(f"- Vocab kept: {len(set().union(*transactions_pruned)) if transactions_pruned else 0}\n")
    f.write(f"- Frequent itemsets: {len(fis_df)}; Rules: {len(rules_df)}\n")
    f.write("- Guidance: Included domain synonyms (e.g., masjid->amenity=mosque) as osm_hint:* items to bias useful patterns.\n")
    if len(rules_df) > 0:
        top = rules_df.head(5).to_dict(orient="records")
        f.write(f"- Top rules (by lift/conf): {json.dumps(top, ensure_ascii=False)}\n")
    else:
        f.write("- No rules met the thresholds; consider lowering MIN_SUP/MIN_CONF or expanding data.\n")
print("Wrote Markdown report to:", report_md_path)

Wrote Markdown report to: docs/reports\patterns_mining_20250815.md


C:\Users\stran\AppData\Local\Temp\ipykernel_4952\4092089262.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  report_md_path = os.path.join(REPORTS_DIR, f"patterns_mining_{datetime.utcnow().strftime('%Y%m%d')}.md")


In [16]:
print("Notebook 02_patternmining_cleaneddata_from_ASA.ipyn complete.")
print("- frequent_itemsets: results/frequent_itemsets.csv + .json")
print("- association_rules: results/association_rules.csv + .json")
print("- report: docs/reports/patterns_mining_YYYYMMDD.md")
print("Next: Next Notebook :  will prepare embeddings and train/test splits, and fuse mined patterns into features.")

Notebook 02_patternmining_cleaneddata_from_ASA.ipyn complete.
- frequent_itemsets: results/frequent_itemsets.csv + .json
- association_rules: results/association_rules.csv + .json
- report: docs/reports/patterns_mining_YYYYMMDD.md
Next: Next Notebook :  will prepare embeddings and train/test splits, and fuse mined patterns into features.
